In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, sum as spark_sum

In [0]:
SOURCE_TABLE = "workspace.bronze.historical_hourly"

TARGET_TABLE = "workspace.silver.historical_hourly"

In [0]:
# Reading Bronze table
bronze_df = spark.table(SOURCE_TABLE)

In [0]:
# Remove Duplicates

silver_df = bronze_df.dropDuplicates()

In [0]:
# convert time stamp

silver_df = (
    silver_df
    .withColumn(
        "date",
        to_timestamp("date")
    )
    .withColumn(
        "ingestion_timestamp",
        to_timestamp("ingestion_timestamp")
    )
)

In [0]:
# Remove Bad Records

silver_df = silver_df.filter(
    col("city_name").isNotNull()
)

silver_df = silver_df.filter(
    col("date").isNotNull()
)

silver_df = silver_df.filter(
    col("temperature").isNotNull()
)

In [0]:
# Clean text

silver_df = (
    silver_df
    .withColumn(
        "city_name",
        trim(col("city_name"))
    )
    .withColumn(
        "country_code",
        upper(trim(col("country_code")))
    )
)

In [0]:
# Adding col

silver_df = (
    silver_df
    .withColumn(
        "year",
        year("date")
    )
    .withColumn(
        "month",
        month("date")
    )
    .withColumn(
        "day",
        dayofmonth("date")
    )
    .withColumn(
        "hour",
        hour("date")
    )
    .withColumn(
        "day_of_week",
        date_format("date", "EEEE")
    )
)

In [0]:
# Column Rename

silver_df = (
    silver_df
    .withColumnRenamed(
        "humidity_relative",
        "humidity"
    )
    .withColumnRenamed(
        "temperature_realfeel",
        "realfeel_temperature"
    )
    .withColumnRenamed(
        "temperature_dew_point",
        "dew_point_temperature"
    )
)

In [0]:
# Find all null columns
total_rows = silver_df.count()

null_counts = silver_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
])

null_counts_dict = null_counts.first().asDict()

# Store those column names in a list
all_null_columns = [
    column
    for column, null_count in null_counts_dict.items()
    if null_count == total_rows
]

print("Columns with all NULL values:")
print(all_null_columns)

# Drop all null column

silver_df = silver_df.drop(*all_null_columns)

print(f"Removed {len(all_null_columns)} columns.")

for column in all_null_columns:
    print(column)

In [0]:
# Validation
print(f"Rows : {silver_df.count()}")

print(f"Columns : {len(silver_df.columns)}")

silver_df.printSchema()

display(silver_df.limit(10))

In [0]:
# Write Silver layer

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)